In [ ]:
import cv2
import numpy as np

In [ ]:
def make_cloak(frame, background):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)       
    # Red is at both ends of the colour scale, so we check two ranges
    mask1 = cv2.inRange(hsv, (0, 120, 70), (10, 255, 255))
    mask2 = cv2.inRange(hsv, (170, 120, 70), (180, 255, 255))
    red_mask = mask1 | mask2                            
    red_mask = cv2.dilate(red_mask, np.ones((5, 5), np.uint8), iterations=2)   
    output = frame.copy()
    output[red_mask > 0] = background[red_mask > 0]         return output

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

def start_camera():
    js = Javascript("""
    var video, div = null, stream, captureCanvas, imgElement, labelElement;
    var pendingResolve = null, shutdown = false;

    function removeDom() {
      stream.getVideoTracks()[0].stop();
      video.remove(); div.remove();
      video = null; div = null; stream = null;
      imgElement = null; captureCanvas = null; labelElement = null;
    }
    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8);
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }
    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML = '<span style="color: red; font-weight: bold;">Click here or on the video to stop</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);
      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      stream = await createDom();
      if (label != "") { labelElement.innerHTML = label; }
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }
      var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
      shutdown = false;
      return {'img': result};
    }
    """)
    display(js)

def get_frame(status_text, picture_to_show):
    return eval_js('stream_frame("{}", "{}")'.format(status_text, picture_to_show))

def browser_to_opencv(js_reply):
    jpeg_bytes = b64decode(js_reply.split(",")[1])
    return cv2.imdecode(np.frombuffer(jpeg_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)

def opencv_to_browser(picture):
    ok, buffer = cv2.imencode(".jpg", picture)
    return "data:image/jpeg;base64," + b64encode(buffer.tobytes()).decode("utf-8")

In [ ]:
start_camera()
status = "Move OUT of the frame and stay still"
picture_to_show = ""
frame_count = 0
background = None

while True:
    reply = get_frame(status, picture_to_show)
    if not reply:                                   # you clicked stop
        break

    frame = browser_to_opencv(reply["img"])
    frame_count = frame_count + 1

    if frame_count <= 30:                           # first 30 frames: save the background
        background = frame.copy()
        output = frame
    else:                                           # after that: turn the cloak on
        status = "Cloak is ON - show your red cloth"
        output = make_cloak(frame, background)

    picture_to_show = opencv_to_browser(output)

print("Stopped.")